In [7]:
pip install torch torchvision opencv-python matplotlib numpy pandas pillow albumentations tqdm

  Using cached albumentations-2.0.8-py3-none-any.whl.metadata (43 kB)
  Using cached albucore-0.0.24-py3-none-any.whl.metadata (5.3 kB)
  Using cached opencv_python_headless-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached albumentations-2.0.8-py3-none-any.whl (369 kB)
Using cached albucore-0.0.24-py3-none-any.whl (15 kB)
Using cached opencv_python_headless-4.13.0.92-cp37-abi3-win_amd64.whl (40.1 MB)

   ---------------------------------------- 0/3 [opencv-python-headless]

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\Users\\Puneeth Ram\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages\\cv2\\cv2.pyd'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
pip install -U albumentations


  Using cached albumentations-2.0.8-py3-none-any.whl.metadata (43 kB)
  Using cached albucore-0.0.24-py3-none-any.whl.metadata (5.3 kB)
  Using cached opencv_python_headless-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached albumentations-2.0.8-py3-none-any.whl (369 kB)
Using cached albucore-0.0.24-py3-none-any.whl (15 kB)
Using cached opencv_python_headless-4.13.0.92-cp37-abi3-win_amd64.whl (40.1 MB)

   ---------------------------------------- 0/3 [opencv-python-headless]

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\Users\\Puneeth Ram\\AppData\\Local\\Programs\\Python\\Python313\\Lib\\site-packages\\cv2\\cv2.pyd'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import os
import cv2
import torch
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch.nn as nn
import torch.optim as optim

from PIL import Image
from tqdm import tqdm

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

import albumentations as A

ModuleNotFoundError: No module named 'albumentations'

In [11]:
characters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"

char_to_idx = {
    char: idx + 1
    for idx, char in enumerate(characters)
}

idx_to_char = {
    idx: char
    for char, idx in char_to_idx.items()
}

vocab_size = len(characters) + 1

print(vocab_size)

37


In [ ]:
def preprocess_image(image_path):

    image = cv2.imread(
        image_path,
        cv2.IMREAD_GRAYSCALE
    )

    image = cv2.resize(
        image,
        (IMAGE_WIDTH, IMAGE_HEIGHT)
    )

    image = image / 255.0

    return image

In [ ]:
sample_image = preprocess_image(
    "sample.jpg"
)

plt.figure(figsize=(12,4))

plt.imshow(sample_image, cmap='gray')

plt.title("Processed OCR Image")

plt.axis("off")

plt.show()

In [ ]:
transform = A.Compose([

    A.Rotate(
        limit=5,
        p=0.5
    ),

    A.RandomBrightnessContrast(
        p=0.5
    ),

    A.GaussNoise(
        p=0.3
    ),

    A.MotionBlur(
        p=0.2
    )
])

In [ ]:
class OCRDataset(Dataset):

    def __init__(
        self,
        image_paths,
        labels
    ):

        self.image_paths = image_paths
        self.labels = labels

    def __len__(self):

        return len(self.image_paths)

    def __getitem__(self, idx):

        image = preprocess_image(
            self.image_paths[idx]
        )

        image = torch.tensor(
            image,
            dtype=torch.float32
        ).unsqueeze(0)

        label = self.labels[idx]

        encoded = [
            char_to_idx[c]
            for c in label
        ]

        encoded = torch.tensor(encoded)

        return image, encoded

In [ ]:
class CNNEncoder(nn.Module):
            nn.BatchNorm2d(64),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(128),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                128,
                256,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(256),

            nn.ReLU()
        )

    def forward(self, x):

        return self.features(x)

In [ ]:
class BiLSTM(nn.Module):

    def __init__(self):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=2048,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            batch_first=True
        )

    def forward(self, x):

        output, _ = self.lstm(x)

        return output

In [ ]:
class CRNN(nn.Module):
        )

    def forward(self, x):

        features = self.encoder(x)

        b, c, h, w = features.size()

        features = features.permute(
            0,
            3,
            1,
            2
        )

        features = features.reshape(
            b,
            w,
            c * h
        )

        sequence = self.rnn(features)

        output = self.fc(sequence)

        return output

In [ ]:
criterion = nn.CTCLoss(
    blank=0
)

In [ ]:
model = CRNN(
    num_classes=vocab_size
)

In [ ]:
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    patience=2
)

In [ ]:
for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for images, labels in train_loader:

        outputs = model(images)

        outputs = outputs.log_softmax(2)

        outputs = outputs.permute(1, 0, 2)

        input_lengths = torch.full(
            size=(images.size(0),),
            fill_value=outputs.size(0),
            dtype=torch.long
        )

        target_lengths = torch.tensor(
            [len(label) for label in labels]
        )

        targets = torch.cat(labels)

        loss = criterion(
            outputs,
            targets,
            input_lengths,
            target_lengths
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1} Loss: {total_loss:.4f}"
    )

In [ ]:
def decode_predictions(predictions):

    predictions = predictions.argmax(2)

    decoded_text = []

    for pred in predictions:

        text = ""

        previous = -1

        for p in pred:

            p = p.item()

            if p != previous and p != 0:

                text += idx_to_char[p]

            previous = p

        decoded_text.append(text)

    return decoded_text

In [ ]:
with torch.no_grad():

    predictions = model(images)

    decoded = decode_predictions(predictions)

    print(decoded)

NameError: name 'model' is not defined

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(loss_history)

plt.title("OCR Training Loss")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.show()

NameError: name 'loss_history' is not defined

<Figure size 800x500 with 0 Axes>